# 04 — Network state and the graph-model gate

**Question:** does the state of the network around a train (all trains, measured before *t* − 2 min) improve the champion XGBoost model, and is the remaining gain large enough to justify a graph neural network?

1. The network features on the validation month
2. Headline results and paired comparisons
3. Where the network features help
4. What the new champion uses: feature importance
5. Findings and decisions

**Feature sets** (XGBoost, same settings and refit as the champion):

- `full`: the previous champion (current state, timetable, context, prior, history, slack);
- `full_network`: + 13 features over the last 30 min: events, mean delay and share > 3 min late at the current, next and target stations; delay gained on the next segment; network-wide mean delay and share late;
- `full_network_plus`: + 20 features: **trend** (10- and 60-min windows), **corridor** (all stations and segments of the train's route to the target, incl. `net_cor_gain_sum`, the sum of what other trains just gained on each segment) and **train ahead** (the last train that left the current station for the same next station: headway, its delay, its gain if already arrived).

Validation month May 2026, common subset, MAE in min of Δd, day-clustered 95 % CIs; `diff` negative = better.

In [ ]:
# 0. Setup: validation predictions of full, full_network and full_network_plus, with breakdown columns
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import xgboost as xgb

from swissdelay import config
from swissdelay.evaluation.metrics import evaluate, mae_diff_ci
from swissdelay.features.build import FEATURE_GROUPS
from swissdelay.features.network import NETWORK_FEATURES, NETWORK_PLUS_FEATURES
from swissdelay.viz import GRAY, SERIES, use_style

use_style()
pd.set_option("display.float_format", "{:,.3f}".format)
P = config.PROCESSED
KEYS = ["trip_key", "stop_seq", "horizon_min"]

con = duckdb.connect()
preds = con.sql(f"""
SELECT a.*, b.pred_xgb_norefit_full_network_plus, b.pred_xgb_full_network_plus,
       f.hour, f.station_id,
       n.net_cur_n, n.net_all_mean_delay, n.net_cor_gain_sum, n.net_ahead_headway_min
FROM read_parquet('{P}/tabular_predictions_valid_network.parquet') a
JOIN read_parquet('{P}/tabular_predictions_valid.parquet') b USING (trip_key, stop_seq, horizon_min)
JOIN read_parquet('{P}/features_2026-05.parquet') f USING (trip_key, stop_seq, horizon_min)
LEFT JOIN read_parquet('{P}/network_2026-05.parquet') n USING (trip_key, stop_seq, horizon_min)
""").df()
for c in ("in_common_subset", "is_disruption_day"):
    preds[c] = preds[c].astype(bool)
common = preds[preds["in_common_subset"]].copy()
SETS = {"full": "pred_xgb_full", "full_network": "pred_xgb_full_network",
        "full_network_plus": "pred_xgb_full_network_plus"}
print(f"{len(preds):,} rows, {len(common):,} in the common subset, "
      f"{preds['net_cur_n'].isna().mean():.2%} without network features")

## 1. The network features on the validation month

In [ ]:
# 1.1 Coverage and typical values (validation month, one row per point and horizon)
net = con.sql(f"SELECT * FROM read_parquet('{P}/network_2026-05.parquet')").df()
feats = NETWORK_FEATURES + NETWORK_PLUS_FEATURES
pd.DataFrame({
    "group": ["network" if f in NETWORK_FEATURES else "network_plus" for f in feats],
    "available": [net[f].notna().mean() for f in feats],
    "nonzero": [(net[f].fillna(0) != 0).mean() for f in feats],
    "p10": [net[f].quantile(0.1) for f in feats],
    "median": [net[f].median() for f in feats],
    "p90": [net[f].quantile(0.9) for f in feats],
}, index=feats)

> **Finding 1.1 (coverage).**
> - Station summaries exist for **96–97 %** of points (the other 3–4 % saw no measured event at that station in 30 min). Typical values are small: median 10 events in 30 min at the current station, mean delay 0.6 min, and 0 % of trains more than 3 min late for the median point.
> - The **next segment is rarely observed**: at least one train ran it in the last 30 min for 68 % of points (median 1 train), and in the last 10 min for only 23 %. The **corridor** to the target is better covered: 86 % of points have at least one observed segment, median 3 observations, and 90 % of segments seen for the median point.
> - A **train ahead** within 60 min exists for 92 % of points: median headway 30 min, 10 % within 9 min. It has already arrived at the next station for 82 %.
> - Trains typically **recover** on segments: the median gain is −0.7 min per segment and −1.8 min summed over the corridor (timetable running-time reserves). The network-wide mean delay sits around 1.0 min (p10–p90: 0.76–1.41), with 6 % of trains more than 3 min late.

In [ ]:
# 1.2 Corridor signal: Δd by what other trains just gained on the way to the target (15 min, deciles)
g15 = common[common["horizon_min"] == 15].copy()
g15["cor_bucket"] = pd.qcut(g15["net_cor_gain_sum"], 10, duplicates="drop")
cor = g15.groupby("cor_bucket", observed=True).agg(
    n=("delta_min", "size"), corridor_gain=("net_cor_gain_sum", "median"),
    delta_median=("delta_min", "median"), pred_full=("pred_xgb_full", "median"),
    pred_network_plus=("pred_xgb_full_network_plus", "median"))
nan = g15["net_cor_gain_sum"].isna()
print(f"No corridor observation: {nan.mean():.1%} of points (median Δd {g15.loc[nan, 'delta_min'].median():.2f})")
cor

In [ ]:
# 1.3 Chart: median Δd and median predictions by corridor gain decile (15 min)
r = cor.reset_index(drop=True).melt(id_vars=["corridor_gain"], value_vars=["delta_median", "pred_full", "pred_network_plus"],
                                    var_name="series", value_name="minutes")
fig = px.line(r, x="corridor_gain", y="minutes", color="series", markers=True,
              color_discrete_map={"delta_median": GRAY, "pred_full": SERIES[1], "pred_network_plus": SERIES[0]},
              title="Δd vs delay just gained by other trains on the route to the target (15 min)")
fig.update_layout(xaxis_title="corridor gain sum, decile median (min)", yaxis_title="median (min)", legend_title=None)
fig.show()

> **Finding 1.2–1.3 (corridor signal, 15 min).**
> - **The median Δd rises steadily with what other trains just gained on the route**: from −1.18 min in the lowest decile (corridor gain sum ≈ −3.4 min) to −0.47 min in the highest (≈ +0.8 min).
> - The slope is far below 1:1: about 0.7 min of Δd across a 4-min range of corridor gain. Other trains are a noisy proxy for this train's own run.
> - **Without network features, `full` misses this**: its median prediction stays between −1.04 and −0.74 min and is too optimistic in the top decile (−0.74 vs −0.47 observed). `full_network_plus` follows the observed pattern (−1.27 to −0.54).
> - 13.6 % of points have no corridor observation.

## 2. Headline results and paired comparisons

In [ ]:
# 2.1 MAE of the three feature sets (with and without refit) vs the historical baseline
MODELS = ["pred_historical"]
for col in SETS.values():
    MODELS += [col.replace("pred_xgb_", "pred_xgb_norefit_"), col]
res = evaluate(common, MODELS, reference="pred_historical")
display(res.pivot(index="model", columns="horizon_min", values="mae").loc[[m.removeprefix("pred_") for m in MODELS]])
res[["horizon_min", "model", "mae", "mae_lo", "mae_hi", "p90_abs_err", "diff_vs_ref", "diff_lo", "diff_hi"]]

In [ ]:
# 2.2 Paired comparisons on the same points (day-clustered CI of the MAE difference)
def paired(df, a, b, label, by=None):
    keys = ["horizon_min"] + ([by] if by else [])
    rows = []
    for k, g in df.groupby(keys, observed=True):
        y, day = g["delta_min"].to_numpy(), g["operating_day"].to_numpy()
        err_a, err_b = np.abs(y - g[a].to_numpy()), np.abs(y - g[b].to_numpy())
        d, lo, hi = mae_diff_ci(day, err_a, err_b)
        rows.append(dict(zip(keys, k if isinstance(k, tuple) else (k,)))
                    | {"comparison": label, "n": len(g), "mae_ref": err_b.mean(), "diff": d, "lo": lo,
                       "hi": hi, "rel_%": 100 * d / err_b.mean(), "significant": hi < 0 or lo > 0})
    return pd.DataFrame(rows)

comp = pd.concat([
    paired(common, SETS["full_network"], SETS["full"], "network vs full"),
    paired(common, SETS["full_network_plus"], SETS["full"], "network_plus vs full"),
    paired(common, SETS["full_network_plus"], SETS["full_network"], "network_plus vs network"),
    paired(common, "pred_xgb_norefit_full_network_plus", "pred_xgb_norefit_full", "network_plus vs full (no refit)"),
])
comp.round(4)

In [ ]:
# 2.3 Chart: paired MAE differences with 95 % CI
r = comp.assign(plus=comp["hi"] - comp["diff"], minus=comp["diff"] - comp["lo"])
r = r[~r["comparison"].str.contains("no refit")]
fig = px.bar(r, x="horizon_min", y="diff", color="comparison", barmode="group", error_y="plus",
             error_y_minus="minus", color_discrete_sequence=[SERIES[1], SERIES[0], SERIES[2]],
             title="Gain from network features (paired MAE difference, negative = better)")
fig.update_layout(xaxis=dict(type="category", title="horizon (min)"), yaxis_title="Δ MAE (min)", legend_title=None)
fig.show()

> **Finding 2.1–2.3 (headline).**
> - **`full_network_plus` reaches 0.701 / 0.843 / 1.005 min**: −20.6 / −21.4 / −21.4 % vs the historical baseline, compared with −18.3 / −19.9 / −20.3 % for `full`.
> - Paired differences, all significant at every horizon:
>   - `network` vs `full`: −0.014 / −0.013 / −0.009 min;
>   - `network_plus` vs `full`: −0.019 / −0.017 / −0.014 min (−2.7 / −2.0 / −1.4 %);
>   - `network_plus` vs `network`: −0.005 / −0.004 / −0.006 min.
> - The gain holds without the refit step (−0.019 / −0.019 / −0.014). The tail improves too: the 90th-percentile absolute error drops from 1.51 / 1.80 / 2.11 to 1.47 / 1.75 / 2.06 min.
> - **Diminishing returns**: the first 13 network features bring about three quarters of the total gain, and the next 20 the remaining quarter.

## 3. Where the network features help

Paired difference `full_network_plus` − `full` (champion before the network features) by stratum. `rel_%` = difference relative to the MAE of `full` in that stratum.

In [ ]:
# 3.1 Strata
common["d0_bucket"] = pd.cut(common["d0_min"], [-99, 1, 3, 10, 999], right=False,
                             labels=["on time < 1", "minor 1-3", "moderate 3-10", "major > 10"])
common["traffic"] = pd.cut(common["net_cur_n"], [-1, 10, 30, 60, 1e9],
                           labels=["≤ 10 events", "11-30", "31-60", "> 60"])
common["period"] = pd.cut(common["hour"], [-1, 5, 8, 15, 18, 24],
                          labels=["night 0-5", "morning peak 6-8", "day 9-15", "evening peak 16-18", "late 19-23"])
common["network_delay"] = pd.qcut(common["net_all_mean_delay"], 4, labels=["Q1 calm", "Q2", "Q3", "Q4 delayed"])
common["ahead"] = pd.cut(common["net_ahead_headway_min"], [0, 5, 15, 60], labels=["≤ 5 min", "5-15", "15-60"])
common["ahead"] = common["ahead"].cat.add_categories("none").fillna("none")
for col in ["d0_bucket", "traffic", "period", "network_delay", "ahead", "category", "is_disruption_day"]:
    print(col, common[common["horizon_min"] == 15][col].value_counts(sort=False).to_dict())

In [ ]:
# 3.2 Paired gain by stratum (relative %, significant marked with *)
def where(by):
    r = paired(common, SETS["full_network_plus"], SETS["full"], "network_plus vs full", by=by)
    r["cell"] = r["rel_%"].map("{:+.1f}".format) + np.where(r["significant"], "*", "")
    return r

STRATA = ["d0_bucket", "traffic", "period", "network_delay", "ahead", "category", "is_disruption_day"]
by = {s: where(s) for s in STRATA}
for s in STRATA:
    display(by[s].pivot(index=s, columns="horizon_min", values="cell")
            .join(by[s][by[s]["horizon_min"] == 15].set_index(s)[["n", "mae_ref"]]))

In [ ]:
# 3.3 Chart: relative gain by current delay and by network-wide delay
r = pd.concat([by["d0_bucket"].rename(columns={"d0_bucket": "stratum"}).assign(by="current delay"),
               by["network_delay"].rename(columns={"network_delay": "stratum"}).assign(by="network-wide delay")])
r["stratum"] = r["stratum"].astype(str)
fig = px.bar(r, x="stratum", y="rel_%", color="horizon_min", barmode="group", facet_col="by",
             color_discrete_sequence=SERIES, title="Relative MAE change, full_network_plus vs full (%)")
fig.update_xaxes(matches=None, title=None)
fig.update_layout(yaxis_title="Δ MAE (%)", legend_title="horizon")
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("by=", "")))
fig.show()

> **Finding 3.1–3.3 (where the network features help; relative change in MAE vs `full`, 15 / 30 / 60 min).**
> - **The gain is broad, not concentrated.** It is significant in almost every stratum, typically −1.5 to −3 %: every category, every period of the day except the night, and every level of current delay.
> - **Largest gains:**
>   - a **train ahead that left ≤ 5 min earlier**: −4.9 / −4.3 / −1.8 % (7,437 points). Close following is where interactions between trains matter;
>   - **major current delays** at 15 min: −5.4 % (only 1,654 points; not significant at 30 and 60 min);
>   - **no train ahead within 60 min**: −3.5 % at 15 min. Knowing the line is quiet is also information;
>   - RE at 30 min: −3.1 %.
> - **Smallest gains:**
>   - the night: −0.7 / −0.4 / −0.3 %, mostly not significant;
>   - the busiest stations (more than 60 events in 30 min) at 60 min: −0.4 %, not significant;
>   - the most delayed quarter of network states at 60 min: −0.9 %.
> - **The gain does not grow with congestion.** It is about the same in calm and delayed network states, and at quiet and busy stations. At the 60-min horizon it is even smaller in the most delayed states.
>   - Delay propagation in congested situations is exactly what a graph model would aim to capture. These summaries do not show that this is where the extra information lies.
> - Disruption day: −2.7 / −2.6 / −1.7 %, similar or slightly larger. This is one day only.

## 4. What the new champion uses

Gain importance of `models/xgb_full_network_plus_h*.json`, with the network features split into their four families.

In [ ]:
# 4.1 Gain importance by feature group
group_of = {f: g for g, fs in FEATURE_GROUPS.items() for f in fs}
group_of |= dict.fromkeys(NETWORK_FEATURES, "net: 30 min")
for f in NETWORK_PLUS_FEATURES:
    group_of[f] = ("net: trend" if f.endswith(("_10", "_60")) else
                   "net: corridor" if f.startswith("net_cor") else "net: train ahead")
imp = []
for h in config.HORIZONS_MIN:
    booster = xgb.Booster()
    booster.load_model(config.ROOT / "models" / f"xgb_full_network_plus_h{h}.json")
    gain = booster.get_score(importance_type="total_gain")
    total = sum(gain.values())
    imp += [{"horizon_min": h, "feature": f, "group": group_of.get(f, "?"), "share": g / total}
            for f, g in gain.items()]
imp = pd.DataFrame(imp)
imp.pivot_table(index="group", columns="horizon_min", values="share", aggfunc="sum").sort_values(15, ascending=False).mul(100).round(1)

In [ ]:
# 4.2 Top 20 features, and the top network features
display(imp.pivot_table(index=["feature", "group"], columns="horizon_min", values="share")
          .sort_values(60, ascending=False).head(20).mul(100).round(1))
imp[imp["group"].str.startswith("net")].pivot_table(index=["feature", "group"], columns="horizon_min", values="share") \
    .sort_values(60, ascending=False).head(10).mul(100).round(1)

In [ ]:
# 4.3 Chart: top 15 features at 60 min
top = imp[imp["horizon_min"] == 60].nlargest(15, "share").sort_values("share")
fig = px.bar(top, x="share", y="feature", color="group", orientation="h",
             title="XGBoost full_network_plus (60 min): share of total gain by feature")
fig.update_layout(xaxis_tickformat=".0%", xaxis_title=None, yaxis_title=None, legend_title=None)
fig.show()

> **Finding 4.1–4.3 (gain importance).**
> - **The network features carry 15 / 18 / 14 % of the total gain at 15 / 30 / 60 min.** The prior and the current delay still dominate: 58 / 62 / 70 % together.
> - By family:
>   - 30-min summaries: 5.5 / 6.1 / 4.4 %;
>   - corridor: 4.0 / 6.1 / 4.9 %;
>   - trend: 3.3 / 3.5 / 3.0 %;
>   - train ahead: 2.0 / 1.8 / 1.3 %.
> - The top network features are the corridor mean delay, the corridor event count, the target-station mean delay and the train-ahead headway, each only about 1–2 %. **No single network feature is decisive**: the information is spread thinly over many weak signals.
> - Slack loses some importance compared with `full` (13.7 % at 15 min, down from 20 %). Part of what slack captured, how much recovery is possible on the way, is now also seen in what other trains actually achieved on the corridor.

## 5. Findings and decisions

### Results (validation, common subset, MAE in min at 15 / 30 / 60 min)

| Model (XGBoost, refit) | MAE | vs historical | vs `full` (paired) |
|---|---|---|---|
| Historical median | 0.883 / 1.073 / 1.279 | reference | |
| `full` (previous champion) | 0.720 / 0.860 / 1.020 | −18.3 / −19.9 / −20.3 % | reference |
| `full_network` (+13 network features) | 0.706 / 0.847 / 1.011 | −20.0 / −21.1 / −20.9 % | −0.014 / −0.013 / −0.009 |
| **`full_network_plus` (+20 more)** | **0.701 / 0.843 / 1.005** | **−20.6 / −21.4 / −21.4 %** | **−0.019 / −0.017 / −0.014** |

### Answer to the question of this notebook

- **Does the network state help?** Yes, significantly at every horizon, in almost every stratum and with or without the refit. The effect is small: 1.4–2.7 %.
- **Is the remaining gain large enough to justify a graph neural network?** On these results, not as a production choice.
  - Each additional layer of network information brings less: −0.014 min, then −0.005 min.
  - The gain is spread thinly over many weak signals.
  - It does not grow in congested or delayed situations, where propagation effects should matter most.
  - A graph model would compete for a remaining gain of the order of a hundredth of a minute.
- **Where could a graph model still add something?**
  - Interactions between closely following trains: the largest gain is for headways ≤ 5 min.
  - Patterns that summaries cannot express, such as the order and timing of events along the corridor, and conflicts at junctions.
  - This is a research question, not an expected large gain.

### Decisions

| # | Topic | Decision | Evidence |
|---|---|---|---|
| 1 | Champion | **XGBoost `full_network_plus`**, refit on Aug–Apr, same settings as before | 2.1–2.3 |
| 2 | Network features | Kept: all three steps are significant; leakage tested (no event after the minute of *t* − 2 min is used) | 2.2, `tests/test_network.py` |
| 3 | Graph-model gate | **Not justified by the expected gain.** A graph model is run as a **bounded research experiment**, not as the production path. **Bar to beat: `full_network_plus` (paired CI on validation)** | 2.2, 3.2, 4.1 |
| 4 | Graph-model design | Small subgraph per prediction point (the corridor to the target and its neighbouring stations), with the same leakage cut-off. A graph transformer with the tabular features as a global token, and the same model without the graph as a control. 15-min horizon first | 3.2 (largest gains for close interactions) |
| 5 | Test month | Unchanged: June is used once, at the end, for the final champion and, if it beats the bar on validation, the graph model | — |
| 6 | Tree cap | Keep 6,000 trees: the best iteration is 4,427 at 15 min for `full_network_plus` | training log |